In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier

In [2]:

train_m = pd.read_csv("tourney_train_m.csv")
test_m_2 = pd.read_csv("tourney_test_m.csv")
val_m = pd.read_csv("tourney_val_m.csv")

train_w = pd.read_csv("tourney_train_w.csv")
test_w = pd.read_csv("tourney_test_w.csv")
val_w = pd.read_csv("tourney_val_w.csv")

In [3]:
X = train_m.drop(columns=['Is1Winner','1TeamID','2TeamID','Season'])
y = train_m['Is1Winner']

In [4]:
test_m = pd.concat([test_m_2, val_m], ignore_index=True)

In [5]:
X_test = test_m.drop(columns=['Is1Winner','1TeamID','2TeamID','Season'])
y_test = test_m['Is1Winner']


### Klasyfikacja

In [ ]:
pd.set_option('display.max_columns', None)
print(X.columns)

In [ ]:
# Logistic Regression

pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=5000))
])

param_grid_lr = {
    'model__C': [0.001, 0.01, 0.1, 1, 10],
    'model__penalty': ['l2'],
    'model__solver': ['lbfgs']
}

grid_lr = GridSearchCV(
    pipe_lr,
    param_grid_lr,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_lr.fit(X, y)

print("LR best params:", grid_lr.best_params_)
print("LR best score:", grid_lr.best_score_)

In [ ]:
# Random Forest

rf = RandomForestClassifier(random_state=42)

param_grid_rf = {
    'n_estimators': [200, 500],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid_rf = GridSearchCV(
    rf,
    param_grid_rf,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X, y)

print("RF best params:", grid_rf.best_params_)
print("RF best score:", grid_rf.best_score_)

In [ ]:
# XGBoost

xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

param_grid_xgb = {
    'n_estimators': [300, 600],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_xgb = GridSearchCV(
    xgb,
    param_grid_xgb,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_xgb.fit(X, y)

print("XGB best params:", grid_xgb.best_params_)
print("XGB best score:", grid_xgb.best_score_)

In [ ]:
# lightgbm

lgb = LGBMClassifier(random_state=42)

param_grid_lgb = {
    'n_estimators': [300, 600],
    'max_depth': [-1, 10, 20],
    'learning_rate': [0.01, 0.1],
    'num_leaves': [31, 63],
    'subsample': [0.8, 1.0]
}

grid_lgb = GridSearchCV(
    lgb,
    param_grid_lgb,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_lgb.fit(X, y)

print("LGB best params:", grid_lgb.best_params_)
print("LGB best score:", grid_lgb.best_score_)

In [ ]:
print("LR:", grid_lr.best_score_)
print("RF:", grid_rf.best_score_)
print("XGB:", grid_xgb.best_score_)
print("LGB:", grid_lgb.best_score_)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

def evaluate_model(model, X_val, y_val):
    """
    Ocena już wytrenowanego modelu.
    Rysuje ROC curve i confusion matrix.
    """

    # Predykcje klas
    y_pred = model.predict(X_val)

    # Prawdopodobieństwa do ROC
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_val)[:, 1]
    else:
        y_prob = model.decision_function(X_val)

    # Metryki
    acc = accuracy_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_prob)

    print("Accuracy:", acc)
    print("ROC-AUC:", auc)
    print("\nClassification report:\n")
    print(classification_report(y_val, y_pred))

    # ===== ROC Curve =====
    fpr, tpr, _ = roc_curve(y_val, y_prob)

    plt.figure()
    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.show()

    # ===== Confusion Matrix =====
    cm = confusion_matrix(y_val, y_pred)

    plt.figure()
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title("Confusion Matrix")
    plt.show()

    return acc, auc

In [ ]:
X_val = val_m.drop(columns=['Is1Winner','1TeamID','2TeamID','Season'])
y_val = val_m['Is1Winner']

In [ ]:
evaluate_model(grid_lr, X, y)


In [ ]:
evaluate_model(grid_rf, X_val, y_val)

In [ ]:
evaluate_model(grid_xgb, X_val, y_val)

## Regresja

In [33]:
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=5000))
])

param_grid_lr = {
    'model__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'model__penalty': ['l2'],
    'model__solver': ['lbfgs'],
    "model__class_weight": [None, "balanced"]
}

grid_lr = GridSearchCV(
    pipe_lr,
    param_grid_lr,
    cv=5,
    scoring='neg_brier_score',
    n_jobs=-1
)

grid_lr.fit(X, y)

print("Best params:", grid_lr.best_params_)
print("Best score:", grid_lr.best_score_)

Best params: {'model__C': 0.01, 'model__class_weight': 'balanced', 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
Best score: -0.1962927803655608


In [34]:
probs_lr = grid_lr.best_estimator_.predict_proba(X)
p_team1_win_lr = probs_lr[:,1]

In [35]:
mean_squared_error(y, p_team1_win_lr)

0.1770104143182551

In [36]:
p_clipped_lr = np.where(
    p_team1_win_lr > 0.95, 1,
    np.where(p_team1_win_lr < 0.05, 0, p_team1_win_lr)
)

In [37]:
mean_squared_error(y, p_clipped_lr)

0.17704776591371585

In [38]:
probs_lr_test = grid_lr.best_estimator_.predict_proba(X_test)
p_team1_win_lr_test = probs_lr_test[:,1]

In [39]:
mean_squared_error(p_team1_win_lr_test, y_test)

0.1803243525669806

In [40]:
p_clipped_lr_test = np.where(
    p_team1_win_lr_test > 0.95, 1,
    np.where(p_team1_win_lr_test < 0.05, 0, p_team1_win_lr_test)
)

In [41]:
mean_squared_error(p_clipped_lr_test, y_test)

0.18029131037642127

# XGBoost

In [6]:
pipe_xgb = Pipeline([
    ("clf", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        tree_method="hist"
    ))
])

param_grid_xgb = {
    "clf__n_estimators": [300, 500, 800],
    "clf__learning_rate": [0.01, 0.05, 0.1],
    "clf__max_depth": [3, 5, 7],
    "clf__subsample": [0.7, 0.9, 1.0],
    "clf__colsample_bytree": [0.7, 0.9, 1.0],
    "clf__gamma": [0, 0.1, 0.3],
    "clf__reg_lambda": [1, 5, 10]
}




In [7]:
grid_xgb = GridSearchCV(
    pipe_xgb,
    param_grid_xgb,
    cv=5,
    scoring='neg_brier_score',   # dobre dla probability
    n_jobs=-1,
    verbose=1
)

grid_xgb.fit(X, y)

Fitting 5 folds for each of 2187 candidates, totalling 10935 fits


KeyboardInterrupt: 

In [ ]:
print("Best params:", grid_xgb.best_params_)
print("Best score:", grid_xgb.best_score_)

In [ ]:
probs_xgb = grid_xgb.best_estimator_.predict_proba(X)
p_team1_win_xgb = probs_xgb[:,1]

In [ ]:
mean_squared_error(y, p_team1_win_xgb)

In [ ]:
probs_xgb_test = grid_xgb.best_estimator_.predict_proba(X_test)
p_team1_win_xgb_test = probs_xgb_test[:,1]

In [ ]:
mean_squared_error(y_test, p_team1_win_xgb_test)

In [ ]:
gb = GradientBoostingClassifier()

param_grid_gb = {
    'n_estimators': [200, 400],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 3, 4],
    'subsample': [0.8, 1.0]
}

grid_gb = GridSearchCV(
    gb,
    param_grid_gb,
    cv=5,
    scoring='neg_log_loss',
    n_jobs=-1,
    verbose=1
)

In [ ]:
grid_gb.fit(X, y)

print("Best params:", grid_gb.best_params_)
print("Best score:", grid_gb.best_score_)

In [ ]:
probs_gb = grid_gb.best_estimator_.predict_proba(X)
p_team1_win_gb = probs_gb[:,1]
mean_squared_error(y, p_team1_win_gb)

In [ ]:
rf = RandomForestClassifier(n_jobs=-1)

param_grid_rf = {
    'n_estimators': [300, 600],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}

grid_rf = GridSearchCV(
    rf,
    param_grid_rf,
    cv=5,
    scoring='neg_log_loss',
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X, y)

print("Best params:", grid_rf.best_params_)
print("Best score:", grid_rf.best_score_)

In [ ]:
probs_rf = grid_rf.best_estimator_.predict_proba(X)
p_team1_win_rf = probs_rf[:,1]
mean_squared_error(y, p_team1_win_rf)

In [ ]:
probs_rf_test = grid_rf.best_estimator_.predict_proba(X_test)
p_team1_win_rf_test = probs_rf_test[:,1]
mean_squared_error(y_test, p_team1_win_rf_test)

In [31]:
from lightgbm import LGBMClassifier
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV

lgbm = LGBMClassifier(
    objective='binary',
    n_jobs=-1,
    random_state=52
)

param_dist_lgbm = {
    'n_estimators': randint(100, 300),
    'learning_rate': uniform(0.01, 0.05),
    'num_leaves': randint(20, 50)
}

random_lgbm = RandomizedSearchCV(
    lgbm,
    param_distributions=param_dist_lgbm,
    n_iter=20,
    cv=3,
    scoring='neg_log_loss',
    n_jobs=-1,
    verbose=1,
    random_state=52
)

random_lgbm.fit(X, y,
    eval_set=[(X, y)],
    eval_metric='logloss',
    early_stopping_rounds=20,
    verbose=0
)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


ValueError: 
All the 60 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
60 fits failed with the following error:
Traceback (most recent call last):
  File "/home/joziop/anaconda3/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
TypeError: LGBMClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


In [ ]:
probs_rf = grid_rf.best_estimator_.predict_proba(X)
p_team1_win_rf = probs_rf[:,1]
mean_squared_error(y, p_team1_win_rf)